# 19A3 — Temporal AIA Calibration + Threshold Freeze

Use the frozen Cycle-24 CNN–GRU from 19A2/19A2R.

- Fit Platt calibration **only** on `cycle24_calibration_holdout`.
- Select operating threshold by maximum TSS **only** on `cycle24_threshold_holdout`.
- Do not update model weights.
- Do not touch Cycle-25 or 2026.
- Use manifest `label_48h_final`; ignore embedded NPZ `y`.


In [1]:
from pathlib import Path
import hashlib, json, sys
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss, roc_curve, confusion_matrix, precision_score, recall_score, f1_score

HOME=Path.home()
PREP=HOME/"aia19_cycle24_cache_prep"
TARGET_MAP=PREP/"cycle24_temporal_targets_local_paths.csv.gz"
BASE=HOME/"aia19_cnn_gru_20260917"
MODEL_PATH=BASE/"models"/"cnn_gru_cycle24_final_refit.pt"
NORM_PATH=BASE/"normalisation.json"
DEV_PRED=BASE/"predictions"/"cnn_gru_internal_validation.csv.gz"
OUT=HOME/"aia19_calibration_threshold_20260918"
(OUT/"predictions").mkdir(parents=True,exist_ok=True)

EXPECTED_CHANNELS=["aia94","aia131","aia171","aia193","aia211","aia335"]
IMAGE_SIZE=256; BATCH_SIZE=8; NUM_WORKERS=2; DROPOUT=.30
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE",DEVICE)
print("GPU",torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)


DEVICE cuda
GPU NVIDIA L4


In [2]:
def sha256_file(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()

model_sha=sha256_file(MODEL_PATH)
norm_sha=sha256_file(NORM_PATH)
print("MODEL_SHA256",model_sha)
print("NORMALISATION_SHA256",norm_sha)

rows=pd.read_csv(TARGET_MAP)
cal_df=rows[rows["role"].eq("cycle24_calibration_holdout")].copy()
thr_df=rows[rows["role"].eq("cycle24_threshold_holdout")].copy()
assert len(cal_df)>0 and len(thr_df)>0
assert set(cal_df["target_sample_id"]).isdisjoint(set(thr_df["target_sample_id"]))
assert set(cal_df["region_component_id"]).isdisjoint(set(thr_df["region_component_id"]))
print("Calibration:",len(cal_df),"pos",int(cal_df.label_48h_final.sum()),"prev",float(cal_df.label_48h_final.mean()))
print("Threshold:",len(thr_df),"pos",int(thr_df.label_48h_final.sum()),"prev",float(thr_df.label_48h_final.mean()))

if DEV_PRED.exists():
    d=pd.read_csv(DEV_PRED)
    prev=float(d.y_true.mean())
    pr=float(average_precision_score(d.y_true,d.probability))
    print("Internal validation prevalence:",prev)
    print("Internal validation PR-AUC:",pr)
    print("PR lift:",pr/prev if prev>0 else np.nan)


MODEL_SHA256 11dc35e089101c8b79d2a6ba6f82d02cdb470d583552cad73e6071016c2a2d76
NORMALISATION_SHA256 c7552cebf1629f799936edf41365d0ab0e0eb16ed47207ea90490d501962503e


Calibration: 6253 pos 133 prev 0.021269790500559733
Threshold: 6317 pos 202 prev 0.03197720436916258
Internal validation prevalence: 0.008203559510567298
Internal validation PR-AUC: 0.03695995400882792
PR lift: 4.505355749686277


In [3]:
norm=json.loads(NORM_PATH.read_text())
scale=np.asarray(norm["channel_scale"],np.float32)
mean=np.asarray(norm["channel_mean"],np.float32)
std=np.asarray(norm["channel_std"],np.float32)

class DS(Dataset):
    def __init__(self,df): self.df=df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def _load(self,p):
        with np.load(p,allow_pickle=False) as z:
            x=z["x"]; ch=[str(v) for v in z["channels"].tolist()]
            if x.shape!=(512,512,6) or ch!=EXPECTED_CHANNELS: raise ValueError(p)
            if not np.isfinite(x).all(): raise ValueError("nonfinite "+str(p))
            x=x.astype(np.float32,copy=False)
        x=np.arcsinh(x/scale.reshape(1,1,6))
        x=(x-mean.reshape(1,1,6))/std.reshape(1,1,6)
        t=torch.from_numpy(x).permute(2,0,1).contiguous()
        return F.interpolate(t.unsqueeze(0),size=(IMAGE_SIZE,IMAGE_SIZE),mode="bilinear",align_corners=False).squeeze(0)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        x=torch.stack([self._load(r.local_tminus288),self._load(r.local_tminus192),self._load(r.local_tminus96)])
        return x, torch.tensor(float(r.label_48h_final),dtype=torch.float32), r.target_sample_id, r.region_component_id

class CB(nn.Module):
    def __init__(self,ci,co,d=0):
        super().__init__(); self.net=nn.Sequential(nn.Conv2d(ci,co,3,2,1,bias=False),nn.BatchNorm2d(co),nn.GELU(),nn.Conv2d(co,co,3,1,1,bias=False),nn.BatchNorm2d(co),nn.GELU(),nn.Dropout2d(d) if d else nn.Identity())
    def forward(self,x): return self.net(x)

class FrameCNN(nn.Module):
    def __init__(self,e=256):
        super().__init__(); self.enc=nn.Sequential(CB(6,32,.05),CB(32,64,.05),CB(64,128,.1),CB(128,192,.1),nn.AdaptiveAvgPool2d(1)); self.proj=nn.Sequential(nn.Flatten(),nn.Linear(192,e),nn.GELU(),nn.Dropout(DROPOUT))
    def forward(self,x): return self.proj(self.enc(x))

class Model(nn.Module):
    def __init__(self,e=256,h=192):
        super().__init__(); self.frame=FrameCNN(e); self.gru=nn.GRU(e,h,batch_first=True); self.head=nn.Sequential(nn.LayerNorm(h),nn.Dropout(DROPOUT),nn.Linear(h,1))
    def forward(self,x):
        b,t,c,h,w=x.shape
        z=self.frame(x.reshape(b*t,c,h,w)).reshape(b,t,-1)
        _,hh=self.gru(z)
        return self.head(hh[-1]).squeeze(-1)

model=Model().to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH,map_location=DEVICE))
model.eval()
print("Frozen model loaded.")


Frozen model loaded.


In [4]:
def infer(df,name):
    dl=DataLoader(DS(df),batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=True)
    ys=[]; lg=[]; ids=[]; regs=[]
    with torch.no_grad():
        for bi,(x,y,sid,reg) in enumerate(dl,1):
            logits=model(x.to(DEVICE,non_blocking=True)).cpu().numpy()
            ys.extend(y.numpy().tolist()); lg.extend(logits.tolist()); ids.extend(list(sid)); regs.extend(list(reg))
            if bi%500==0: print(name,bi,"/",len(dl),flush=True)
    o=pd.DataFrame({"target_sample_id":ids,"region_component_id":regs,"y_true":np.asarray(ys,int),"raw_logit":np.asarray(lg,float)})
    o["raw_probability"]=1/(1+np.exp(-o.raw_logit.to_numpy()))
    return o

cal=infer(cal_df,"calibration")
thr=infer(thr_df,"threshold")
cal.to_csv(OUT/"predictions"/"calibration_raw.csv.gz",index=False,compression="gzip")
thr.to_csv(OUT/"predictions"/"threshold_raw.csv.gz",index=False,compression="gzip")


calibration 500 / 782


threshold 500 / 790


In [5]:
def pm(y,p):
    return {"roc_auc":float(roc_auc_score(y,p)),"pr_auc":float(average_precision_score(y,p)),"brier":float(brier_score_loss(y,p)),"log_loss":float(log_loss(y,np.clip(p,1e-7,1-1e-7),labels=[0,1])),"prevalence":float(np.mean(y))}

platt=LogisticRegression(solver="lbfgs",C=1e6,max_iter=1000,random_state=20260918)
platt.fit(cal[["raw_logit"]].to_numpy(),cal.y_true.to_numpy())
coef=float(platt.coef_[0,0]); intercept=float(platt.intercept_[0])
if coef<=0: raise RuntimeError(f"Unexpected Platt slope {coef}")

cal["calibrated_probability"]=platt.predict_proba(cal[["raw_logit"]].to_numpy())[:,1]
cal_rec={"method":"Platt scaling on frozen raw logits","fit_role":"cycle24_calibration_holdout","coefficient":coef,"intercept":intercept,"raw_metrics":pm(cal.y_true,cal.raw_probability),"calibrated_metrics":pm(cal.y_true,cal.calibrated_probability)}
(OUT/"platt_calibrator.json").write_text(json.dumps(cal_rec,indent=2)+"\n")
cal.to_csv(OUT/"predictions"/"calibration_calibrated.csv.gz",index=False,compression="gzip")
print(json.dumps(cal_rec,indent=2))


{
  "method": "Platt scaling on frozen raw logits",
  "fit_role": "cycle24_calibration_holdout",
  "coefficient": 0.5846760189574352,
  "intercept": -3.6728404851652265,
  "raw_metrics": {
    "roc_auc": 0.6841527839205856,
    "pr_auc": 0.03811900155090733,
    "brier": 0.204366078756317,
    "log_loss": 0.5727020749467667,
    "prevalence": 0.021269790500559733
  },
  "calibrated_metrics": {
    "roc_auc": 0.6841527839205856,
    "pr_auc": 0.03811900155090733,
    "brier": 0.020625139637751,
    "log_loss": 0.09874373777229457,
    "prevalence": 0.021269790500559733
  }
}


In [6]:
thr["calibrated_probability"]=platt.predict_proba(thr[["raw_logit"]].to_numpy())[:,1]
y=thr.y_true.to_numpy(int); p=thr.calibrated_probability.to_numpy(float)
fpr,tpr,ths=roc_curve(y,p); tss=tpr-fpr
idxs=np.where(np.isfinite(ths))[0]
j=idxs[np.argmax(tss[idxs])]
threshold=float(ths[j]); yhat=(p>=threshold).astype(int)
tn,fp,fn,tp=confusion_matrix(y,yhat,labels=[0,1]).ravel()
den=((tp+fn)*(fn+tn)+(tp+fp)*(fp+tn))
hss=float(2*(tp*tn-fn*fp)/den) if den else float("nan")
rec={**pm(y,p),"threshold":threshold,"tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp),"tss":float(tss[j]),"hss":hss,"precision":float(precision_score(y,yhat,zero_division=0)),"recall":float(recall_score(y,yhat,zero_division=0)),"f1":float(f1_score(y,yhat,zero_division=0))}
(OUT/"operating_threshold.json").write_text(json.dumps(rec,indent=2)+"\n")
thr["prediction"]=yhat
thr.to_csv(OUT/"predictions"/"threshold_calibrated_predictions.csv.gz",index=False,compression="gzip")
print(json.dumps(rec,indent=2))


{
  "roc_auc": 0.7823895145033718,
  "pr_auc": 0.0903245909115584,
  "brier": 0.03042529373197073,
  "log_loss": 0.13088905893715028,
  "prevalence": 0.03197720436916258,
  "threshold": 0.030438695842933242,
  "tn": 4024,
  "fp": 2091,
  "fn": 44,
  "tp": 158,
  "tss": 0.4402321834799997,
  "hss": 0.07462250000360221,
  "precision": 0.07025344597598933,
  "recall": 0.7821782178217822,
  "f1": 0.12892696858425132
}


In [7]:
protocol={
"status":"TEMPORAL_AIA_CNN_GRU_CYCLE24_PIPELINE_FROZEN_PENDING_INDEPENDENT_CYCLE25_EVALUATION",
"base_model_path":str(MODEL_PATH),"base_model_sha256":model_sha,
"normalisation_path":str(NORM_PATH),"normalisation_sha256":norm_sha,
"channel_order":EXPECTED_CHANNELS,"history_lags_minutes":[-288,-192,-96],
"forecast_horizon_hours":48,"authoritative_label":"label_48h_final","embedded_npz_y_used":False,
"calibration_method":"Platt scaling on frozen raw logits","calibration_role":"cycle24_calibration_holdout",
"platt_coefficient":coef,"platt_intercept":intercept,
"threshold_role":"cycle24_threshold_holdout","threshold_selection_rule":"maximum TSS","frozen_threshold":threshold,
"threshold_holdout_metrics":rec,"cycle25_used":False,"supplementary_2026_used":False,
"model_weights_updated":False,"scientific_clearance":False}
(OUT/"protocol_record.json").write_text(json.dumps(protocol,indent=2)+"\n")
print(json.dumps(protocol,indent=2))


{
  "status": "TEMPORAL_AIA_CNN_GRU_CYCLE24_PIPELINE_FROZEN_PENDING_INDEPENDENT_CYCLE25_EVALUATION",
  "base_model_path": "/home/abmoses2000/aia19_cnn_gru_20260917/models/cnn_gru_cycle24_final_refit.pt",
  "base_model_sha256": "11dc35e089101c8b79d2a6ba6f82d02cdb470d583552cad73e6071016c2a2d76",
  "normalisation_path": "/home/abmoses2000/aia19_cnn_gru_20260917/normalisation.json",
  "normalisation_sha256": "c7552cebf1629f799936edf41365d0ab0e0eb16ed47207ea90490d501962503e",
  "channel_order": [
    "aia94",
    "aia131",
    "aia171",
    "aia193",
    "aia211",
    "aia335"
  ],
  "history_lags_minutes": [
    -288,
    -192,
    -96
  ],
  "forecast_horizon_hours": 48,
  "authoritative_label": "label_48h_final",
  "embedded_npz_y_used": false,
  "calibration_method": "Platt scaling on frozen raw logits",
  "calibration_role": "cycle24_calibration_holdout",
  "platt_coefficient": 0.5846760189574352,
  "platt_intercept": -3.6728404851652265,
  "threshold_role": "cycle24_threshold_holdout"